In [0]:
# Crio o schema responsável pela camada Gold, onde ficarão os dados agregados para análise e consumo analítico.

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.gold
    COMMENT 'Camada Gold com agregações e estruturas analíticas dos preços de combustíveis da ANP.'
""")

print("Schema Gold disponível.")

In [0]:
# A Gold parte exclusivamente da Silver já tratada e aprovada.

nome_tabela_silver = "workspace.silver.revendas_anp"

df_silver = spark.table(nome_tabela_silver)

print("Registros recebidos da Silver:", df_silver.count())
print("Colunas recebidas:", len(df_silver.columns))

In [0]:
from pyspark.sql.functions import (
    col,
    weekofyear,
    year,
    date_add,
    date_trunc
)

# Identifico a semana ISO de cada coleta.
# Para o ano ISO, uso a quinta-feira da própria semana,
# evitando problemas nas semanas que cruzam a virada do ano.
df_gold_base = (
    df_silver
    .withColumn(
        "semana_iso",
        weekofyear(col("data_coleta"))
    )
    .withColumn(
        "ano",
        year(
            date_add(
                date_trunc("week", col("data_coleta")).cast("date"),
                3
            )
        )
    )
)

print("Registros na base Gold:", df_gold_base.count())

In [0]:
display(
    df_gold_base
    .select(
        "ano",
        "semana_iso"
    )
    .distinct()
    .orderBy(
        "ano",
        "semana_iso"
    )
)

In [0]:
from pyspark.sql.functions import (
    count,
    avg,
    min,
    max,
    expr,
    round as spark_round
)

# Agrupo os preços no nível analítico definido para a Gold.
# Mantenho unidade_medida na chave para não misturar litros,
# m³ e botijão de 13 kg na mesma agregação.
df_gold_resumo = (
    df_gold_base
    .groupBy(
        "ano",
        "semana_iso",
        "estado",
        "municipio",
        "produto",
        "unidade_medida",
        "bandeira"
    )
    .agg(
        count("*").alias("quantidade_coletas"),
        avg("preco_revenda").alias("preco_medio"),
        min("preco_revenda").alias("preco_minimo"),
        max("preco_revenda").alias("preco_maximo"),
        expr("percentile_approx(preco_revenda, 0.5)").alias("preco_mediano")
    )
    .withColumn(
        "amplitude_preco",
        col("preco_maximo") - col("preco_minimo")
    )
)

In [0]:
# Padronizo as métricas monetárias em três casas decimais,
# mantendo a mesma regra usada no pipeline local.

colunas_precos = [
    "preco_medio",
    "preco_minimo",
    "preco_maximo",
    "preco_mediano",
    "amplitude_preco"
]

for coluna in colunas_precos:
    df_gold_resumo = df_gold_resumo.withColumn(
        coluna,
        spark_round(col(coluna), 3)
    )

print("Registros Gold:", df_gold_resumo.count())
print(
    "Total de coletas representadas:",
    df_gold_resumo.agg({"quantidade_coletas": "sum"}).collect()[0][0]
)

In [0]:
# Verifico se existe alguma duplicidade na chave analítica da Gold.

chave_gold = [
    "ano",
    "semana_iso",
    "estado",
    "municipio",
    "produto",
    "unidade_medida",
    "bandeira"
]

duplicidades_gold = (
    df_gold_resumo
    .groupBy(*chave_gold)
    .count()
    .filter(col("count") > 1)
)

print("Grupos duplicados na Gold:", duplicidades_gold.count())

In [0]:
# Confirmo que os valores mínimo, médio, mediano e máximo
# mantêm uma relação coerente dentro de cada grupo.

inconsistencias_metricas = (
    df_gold_resumo
    .filter(
        (col("preco_minimo") > col("preco_medio"))
        | (col("preco_medio") > col("preco_maximo"))
        | (col("preco_minimo") > col("preco_mediano"))
        | (col("preco_mediano") > col("preco_maximo"))
        | (col("amplitude_preco") < 0)
    )
)

print(
    "Inconsistências nas métricas:",
    inconsistencias_metricas.count()
)

In [0]:
nome_tabela_gold = "workspace.gold.resumo_precos_semanais"

# Gravo a Gold como tabela Delta gerenciada pelo Unity Catalog.
(
    df_gold_resumo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_tabela_gold)
)

print("Tabela Gold gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros_gold,
            SUM(quantidade_coletas) AS total_coletas,
            COUNT(DISTINCT produto) AS produtos,
            COUNT(DISTINCT unidade_medida) AS unidades_medida
        FROM workspace.gold.resumo_precos_semanais
    """)
)